# March Madness: Data Exploration

Question: Can ML predict the men's March Madness bracket better than just picking the higher seed?

## Load the data

In [10]:
import pandas as pd
import matplotlib.pyplot as plt

folder = "march-machine-learning-mania-2026/"

tourney = pd.read_csv(folder + "MNCAATourneyCompactResults.csv")   
seeds   = pd.read_csv(folder + "MNCAATourneySeeds.csv")       
season  = pd.read_csv(folder + "MRegularSeasonDetailedResults.csv") 
teams   = pd.read_csv(folder + "MTeams.csv")                   
massey  = pd.read_csv(folder + "MMasseyOrdinals.csv")

print("Tournament games:", tourney.shape)
print("Seeds:", seeds.shape)
print("Regular season games:", season.shape)
print("Teams:", teams.shape)
print("Rankings:", massey.shape)

Tournament games: (2585, 8)
Seeds: (2694, 3)
Regular season games: (124529, 34)
Teams: (381, 4)
Rankings: (5865001, 5)


In [11]:
tourney.head()

,Season,DayNum,WTeamID,WScore,LTeamID,LScore,WLoc,NumOT
0,1985,136,1116,63,1234,54,N,0
1,1985,136,1120,59,1345,58,N,0
2,1985,136,1207,68,1250,43,N,0
3,1985,136,1229,58,1425,55,N,0
4,1985,136,1242,49,1325,38,N,0


Each row is one game. `WTeamID` is the winner and `LTeamID` is the loser.
Teams are stored as IDs, so we use `MTeams.csv` to look up their names.

In [12]:
season.head()

,Season,DayNum,WTeamID,WScore,LTeamID,LScore,WLoc,NumOT,WFGM,WFGA,...,LFGA3,LFTM,LFTA,LOR,LDR,LAst,LTO,LStl,LBlk,LPF
0,2003,10,1104,68,1328,62,N,0,27,58,...,10,16,22,10,22,8,18,9,2,20
1,2003,10,1272,70,1393,63,N,0,26,62,...,24,9,20,20,25,7,12,8,6,16
2,2003,11,1266,73,1437,61,N,0,24,58,...,26,14,23,31,22,9,12,2,5,23
3,2003,11,1296,56,1457,50,N,0,18,38,...,22,8,15,17,20,9,19,4,3,23
4,2003,11,1400,77,1208,71,N,0,30,61,...,16,17,27,21,15,12,10,7,1,14


Same idea, but with box score stats. `W` = winner's stats and `L` = loser's stats.

In [13]:
seeds.head()

,Season,Seed,TeamID
0,1985,W01,1207
1,1985,W02,1210
2,1985,W03,1228
3,1985,W04,1260
4,1985,W05,1374


## Check the data

In [14]:
for name, df in [("tourney", tourney), ("seeds", seeds), ("season", season), ("teams", teams)]:
    print(f"{name:8} missing values: {df.isna().sum().sum()}   duplicates: {df.duplicated().sum()}")

tourney  missing values: 0   duplicates: 0
seeds    missing values: 0   duplicates: 0


season   missing values: 0   duplicates: 0
teams    missing values: 0   duplicates: 0


In [15]:
print("Tournament seasons:", tourney.Season.min(), "to", tourney.Season.max())
print("Box score seasons: ", season.Season.min(), "to", season.Season.max())

Tournament seasons: 1985 to 2025
Box score seasons:  2003 to 2026


## Build the modeling table

Make one row per team per game, then average by team and season.

In [16]:
w = pd.DataFrame({"Season": season.Season, "TeamID": season.WTeamID, "OppID": season.LTeamID, "Win": 1,
                  "Points": season.WScore, "OppPoints": season.LScore,
                  "FGM": season.WFGM, "FGA": season.WFGA, "FGM3": season.WFGM3, "FGA3": season.WFGA3,
                  "Reb": season.WOR + season.WDR, "TO": season.WTO,
                  "FTA": season.WFTA, "OR": season.WOR,
                  "OppFGA": season.LFGA, "OppFTA": season.LFTA, "OppOR": season.LOR, "OppTO": season.LTO})

l = pd.DataFrame({"Season": season.Season, "TeamID": season.LTeamID, "OppID": season.WTeamID, "Win": 0,
                  "Points": season.LScore, "OppPoints": season.WScore,
                  "FGM": season.LFGM, "FGA": season.LFGA, "FGM3": season.LFGM3, "FGA3": season.LFGA3,
                  "Reb": season.LOR + season.LDR, "TO": season.LTO,
                  "FTA": season.LFTA, "OR": season.LOR,
                  "OppFGA": season.WFGA, "OppFTA": season.WFTA, "OppOR": season.WOR, "OppTO": season.WTO})

games = pd.concat([w, l])

# Estimated possessions for each team in each game
games["Poss"]    = games.FGA - games.OR + games.TO + 0.44 * games.FTA
games["OppPoss"] = games.OppFGA - games.OppOR + games.OppTO + 0.44 * games.OppFTA

t = games.groupby(["Season", "TeamID"]).sum()       
n_games = games.groupby(["Season", "TeamID"]).size() 

team = pd.DataFrame({
    "WinPct":    t.Win / n_games,
    "PPG":       t.Points / n_games,
    "OppPPG":    t.OppPoints / n_games,
    "FG_Pct":    t.FGM / t.FGA,
    "FG3_Pct":   t.FGM3 / t.FGA3,
    "Reb":       t.Reb / n_games,
    "TO":        t.TO / n_games,
    "OffEff":    100 * t.Points / t.Poss,        # points scored per 100 possessions
    "DefEff":    100 * t.OppPoints / t.OppPoss,  # points allowed per 100 possessions
}).reset_index()
team["PointDiff"] = team.PPG - team.OppPPG

# Strength of schedule: average win % of the opponents each team played
opp_win = team.set_index(["Season", "TeamID"]).WinPct.rename("OppWinPct")
sos = games.join(opp_win, on=["Season", "OppID"]).groupby(["Season", "TeamID"]).OppWinPct.mean()
team = team.join(sos.rename("SOS"), on=["Season", "TeamID"])

# Pomeroy ranking: last one before the tournament starts (day 133)
pom = massey[(massey.SystemName == "POM") & (massey.RankingDayNum <= 133)]
pom = pom.sort_values("RankingDayNum").groupby(["Season", "TeamID"]).OrdinalRank.last()
team = team.join(pom.rename("Rank_POM"), on=["Season", "TeamID"])

seeds["Seed"] = seeds.Seed.str[1:3].astype(int)
team = team.merge(seeds, on=["Season", "TeamID"], how="left")
team.head()

,Season,TeamID,WinPct,PPG,OppPPG,FG_Pct,FG3_Pct,Reb,TO,OffEff,DefEff,PointDiff,SOS,Rank_POM,Seed
0,2003,1102,0.428571,57.250000,57.000000,0.481149,0.375643,21.000000,11.428571,104.924857,105.058058,0.250000,0.530644,160.0,NaN
1,2003,1103,0.481481,78.777778,78.148148,0.486074,0.338710,29.703704,12.629630,112.413589,111.574094,0.629630,0.489779,163.0,NaN
2,2003,1104,0.607143,69.285714,65.000000,0.420362,0.320144,37.500000,13.285714,104.817272,98.474191,4.285714,0.572492,33.0,10.0
3,2003,1105,0.269231,71.769231,76.653846,0.395755,0.364815,36.615385,18.653846,93.961489,101.721040,-4.884615,0.415228,307.0,NaN
4,2003,1106,0.464286,63.607143,63.750000,0.423773,0.346154,36.142857,17.035714,94.540938,95.423928,-0.142857,0.458062,263.0,NaN


One row per tournament game.

In [17]:
df = tourney[tourney.Season >= 2003].copy()
df["TeamA"] = df[["WTeamID", "LTeamID"]].min(axis=1)
df["TeamB"] = df[["WTeamID", "LTeamID"]].max(axis=1)
df["TeamA_Win"] = (df.WTeamID == df.TeamA).astype(int)

df = df.merge(team.add_prefix("A_"), left_on=["Season", "TeamA"], right_on=["A_Season", "A_TeamID"])
df = df.merge(team.add_prefix("B_"), left_on=["Season", "TeamB"], right_on=["B_Season", "B_TeamID"])

features = ["Seed", "WinPct", "PointDiff", "PPG", "OppPPG", "FG_Pct", "FG3_Pct", "Reb", "TO",
            "OffEff", "DefEff", "SOS", "Rank_POM"]
for f in features:
    df[f + "_Diff"] = df["A_" + f] - df["B_" + f]

names = teams.set_index("TeamID").TeamName
df["TeamA_Name"] = df.TeamA.map(names)
df["TeamB_Name"] = df.TeamB.map(names)

feature_cols = []
for f in features:
    feature_cols.append(f + "_Diff")
data = df[["Season", "TeamA_Name", "TeamB_Name"] + feature_cols + ["TeamA_Win"]]

print("Rows (games):", len(data))
print("Features:", len(feature_cols))
data.head()

Rows (games): 1449
Features: 13


,Season,TeamA_Name,TeamB_Name,Seed_Diff,WinPct_Diff,PointDiff_Diff,PPG_Diff,OppPPG_Diff,FG_Pct_Diff,FG3_Pct_Diff,Reb_Diff,TO_Diff,OffEff_Diff,DefEff_Diff,SOS_Diff,Rank_POM_Diff,TeamA_Win
0,2003,TX Southern,UNC Asheville,0.0,0.151724,9.208046,1.593103,-7.614943,0.018262,-0.039433,2.518391,-0.973563,2.653173,-12.376875,-0.104891,-5.0,0
1,2003,Arizona,Vermont,-15.0,0.237685,10.309113,17.421182,7.112069,0.016969,0.009777,4.131773,0.716749,8.258702,-4.352797,0.114559,-141.0,1
2,2003,Arizona St,Memphis,3.0,-0.172414,-1.896552,1.448276,3.344828,0.040251,-0.030989,-3.034483,0.206897,3.754363,5.522507,0.065246,12.0,1
3,2003,C Michigan,Creighton,5.0,-0.085684,-8.805643,0.102403,8.908046,0.005763,-0.008284,-0.198537,4.877743,-4.162118,8.874154,-0.005888,23.0,1
4,2003,California,NC State,-1.0,0.124138,0.324138,2.082759,1.758621,0.009399,0.022444,3.854023,-0.027586,-2.338371,-2.831146,-0.031658,-8.0,1


## The Target

In [18]:
counts = data.TeamA_Win.value_counts()
print(counts)

TeamA_Win
1    725
0    724
Name: count, dtype: int64


## Summary
- Clean data, 1,449 games, 13 features, balanced target
- Baseline to beat: the higher seed wins
- Next steps: train logistic regression, KNN, random forest, and gradient boosting, then compare them to the seed baseline